# Playbook — Manual Homography Calibration (Colab)

Human-in-the-loop pitch calibration for the MVP videos. You click a handful of
point correspondences per video; optical flow fills in every frame between
anchors. Covered frames bypass the automatic keypoint detector entirely
(radar shows **H: MANUAL**).

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

Workflow: setup (cells 1-6) -> click anchors (7-11) -> densify & QA (12-13) ->
run pipeline with your manual H (14-16).

In [ ]:
# 1) Verify GPU
import subprocess
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() if gpu.returncode==0 else 'none (Runtime -> T4 GPU)')

In [ ]:
# 2) System packages
!apt-get install -qq ffmpeg libglib2.0-0 libsm6 libxext6 libxrender-dev

In [ ]:
# 3) Python dependencies
!pip uninstall -qqy opencv-python opencv-python-headless 2>/dev/null
!pip install -q \
    'numpy>=2.0.0,<2.4.0' \
    opencv-python-headless==4.10.0.84 \
    onnxruntime-gpu==1.20.1 \
    tqdm 'requests>=2.32.3' \
    'pydantic>=2.11.7,<2.12.0' pydantic-settings==2.4.0 python-dotenv==1.0.1 \
    'supervision==0.27.0.post2' 'inference==1.2.2' \
    'ultralytics>=8.4.37,<8.5.0' 'lap>=0.5.13,<0.6'
!pip install -q 'transformers>=5.2.0,<5.3.0'
!pip install -q git+https://github.com/roboflow/sports.git@main
print('\nPackages installed.')

In [ ]:
# 3b) OPTIONAL repair: only if Cell 3 ever leaves CUDA unavailable. Then RESTART RUNTIME.
# !pip uninstall -y -q onnxruntime onnxruntime-gpu
# !pip install -q onnxruntime-gpu==1.20.1
# import torch; print('CUDA:', torch.cuda.is_available())

In [ ]:
# 4) Clone the repo (manual-homography branch)
import subprocess
BRANCH = 'claude/setup-gpu-video-testing-JhgUH'
!rm -rf /content/playbook
!git clone --branch {BRANCH} https://github.com/muwafagq/playbook-program.git /content/playbook
import os, sys
os.chdir('/content/playbook')
sys.path.insert(0, '/content/playbook')
!git pull origin {BRANCH}
print('Branch:', subprocess.run(['git','rev-parse','--abbrev-ref','HEAD'],capture_output=True,text=True).stdout.strip())

In [ ]:
# 5) Environment (clean — uses the tuned baseline, DEVICE=cuda)
import shutil, os
shutil.copy('baseline.env', '.env')
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    ROBOFLOW_API_KEY = 'your_key_here'
with open('.env','a') as f:
    f.write(f'\nROBOFLOW_API_KEY={ROBOFLOW_API_KEY}')
from dotenv import dotenv_values
for k,v in dotenv_values('.env').items():
    if v is not None: os.environ[k]=v
os.environ['DEVICE']='cuda'
os.environ['ROBOFLOW_API_KEY']=ROBOFLOW_API_KEY
import torch
print('CUDA:', torch.cuda.is_available(), '| DEVICE:', os.environ.get('DEVICE'))

In [ ]:
# 6) Upload your video, normalize the path
from google.colab import files as colab_files
import os
print('Select your MP4...')
uploaded = colab_files.upload()
src = '/content/' + list(uploaded.keys())[0]
VIDEO = '/content/test_video.mp4'
os.replace(src, VIDEO)
print('Video ready at:', VIDEO)

## Calibration

Click 4-8 correspondences per keyframe. The numbered pitch reference (next cell)
tells you which vertex number is which landmark.

In [ ]:
# 7) Load the calibration tool + show the numbered pitch reference
import sys
for m in list(sys.modules):
    if m.startswith('tools') or m.startswith('geometry'):
        del sys.modules[m]
from tools import manual_calib as mc
mc.show_pitch_reference()

In [ ]:
# 8) Choose keyframes (skip the opening; anchor the rest ~every 20 frames)
keyframes = list(range(172, 396, 20))
print('keyframes:', keyframes, '(', len(keyframes), 'total )')
anchors = {}
i = 0   # progress counter for the click loop below

### Click loop — run cell 9, click, then run cell 10. Repeat for every keyframe.

In **cell 9** a frame + pitch appear. Alternate clicks: a landmark on the FRAME
(top), then the matching numbered vertex on the PITCH (bottom). Do 4-8 pairs,
spread out (corners, box corners, penalty spots, circle/halfway intersections).
Use **Undo last** to fix a misclick. Then run **cell 10** to solve & store it,
which also advances to the next keyframe.

In [ ]:
# 9) SHOW current keyframe (re-run for each anchor)
f = keyframes[i]
print(f'Anchor {i+1}/{len(keyframes)} -> frame {f}')
clk = mc.annotate(VIDEO, f)
clk.show()

In [ ]:
# 10) BUILD current anchor, then advance (re-run after each cell 9)
anchors[f] = clk.build()
print(f'stored anchor frame={f} ({len(clk.image_clicks)} pts). solved H:\n', anchors[f]['H'])
i += 1
if i < len(keyframes):
    print(f'next: re-run cell 9 for frame {keyframes[i]}')
else:
    print('All keyframes done -> go to cell 11.')

In [ ]:
# 11) Densify across all frames between anchors + save the sidecar
dense = mc.build_sidecar(
    VIDEO, anchors,
    out_path='/content/outputs/manual_h.json',
    anchors_path='/content/outputs/manual_h_anchors.json',
)

In [ ]:
# 12) QA: back-project the pitch onto a few frames. Yellow lines should hug the
#     real markings. If a stretch drifts, add an anchor there and re-run cell 11.
for fr in [185, 230, 300, 360]:
    mc.preview_projection(VIDEO, dense, fr)

In [ ]:
# 13) (optional) re-edit later: reload saved anchors instead of re-clicking
# from geometry.manual_h import load_anchors
# anchors = load_anchors('/content/outputs/manual_h_anchors.json')

## Run the pipeline with your manual homography

In [ ]:
# 14) Run, pointing the pipeline at the sidecar
import os, importlib
OUT_DIR = '/content/outputs'; os.makedirs(OUT_DIR, exist_ok=True)
os.environ['H_MANUAL_SIDECAR'] = '/content/outputs/manual_h.json'
with open('.env','a') as fh:
    fh.write('\nH_MANUAL_SIDECAR=/content/outputs/manual_h.json')
import main; importlib.reload(main)
main.main(source_video=VIDEO, out_dir=OUT_DIR, enable_team=True)

In [ ]:
# 15) KPI summary
import json, pandas as pd
with open(OUT_DIR+'/kpi_summary.json') as f: print(json.dumps(json.load(f), indent=2))
df = pd.read_csv(OUT_DIR+'/per_frame_tracks.csv')
print('\nstate counts:'); print(df.drop_duplicates('frame')['homography_state'].value_counts())

In [ ]:
# 16) Download outputs
from google.colab import files as colab_files
for fn in ['annotated.mp4','per_frame_tracks.csv','kpi_summary.json','kpi_summary.csv','manual_h.json','manual_h_anchors.json']:
    p = f'{OUT_DIR}/{fn}'
    if os.path.exists(p): colab_files.download(p)